In [ ]:
# SOme of these file paths are wonky in this and the other API file in this folder, 
# I'm not sure if I changed them all yet so please do not actually run any extraction. 
import requests
import json
import os
import urllib3
import boto3
import io
import gzip
import logging
import pandas as pd
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar

In [ ]:
# Extract data from S3 bucket and prepare for processing
s3_client = boto3.client('s3')
# Defines the current timestamp
today_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
bucket_name = "uncc_traffic_data"

In [ ]:
# Defines the S3 bucket and key for the traffic data file
s3_bucket = "s3-3601-final-project"
s3_key = f"tomtom/traffic/{today_str}speeds1.json.gz"

In [ ]:
api_key = 'KFAr7dIvLb4Fw2ntylN7NFGLqRIJXYxc'
#timestamp = datetime.utcnow()


url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

data_list = []

latitude = [35.307778, 35.313611, 35.311361, 35.309306, 35.306917, 35.293111, 35.294889, 35.299639, 35.287972, 35.301861, 35.311861] # fill in with the found values
longitude = [-80.724583, -80.701944, -80.711222, -80.718056, -80.722639, -80.749361, -80.742111, -80.74575, -80.759472, -80.752333, -80.744]
# Loop through the list of coordinates and fetch traffic data for each location
for i in range(len(latitude)):

        params = {
            'key': api_key,
            'point': f"{latitude[i]},{longitude[i]}",
            'unit': "MPH",
            "openLr": "false"
        }

        response = requests.get(url, params)

        if response.status_code==200:
            data = response.json()
            flow_data = data.get("flowSegmentData", {})

            data_we_want = {
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                "latitude": latitude[i],
                "longitude": longitude[i],
                "current_speed": flow_data.get("currentSpeed"),
                "free_flow_speed": flow_data.get("freeFlowSpeed"),
                "current_delay": flow_data.get("currentDelay"),
                "confidence": flow_data.get("confidence") # for flags of when we should somehow retrain?
            }

            data_list.append(data_we_want)
        else:
            print(f"Error in retrieving data: {response.status_code}")
            
#making data_list into a DataFrame
df = pd.DataFrame(data_list) 
print(df.head())
if not df.empty and "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%Y-%m-%d_%H-%M-%S"
    )
    df["day_of_week"] = df["timestamp"].dt.day_name()

             timestamp   latitude  longitude  current_speed  free_flow_speed  \
0  2026-09-27_15-05-39  35.307778 -80.724583             27               27   
1  2026-09-27_15-05-39  35.313611 -80.701944             40               40   
2  2026-09-27_15-05-39  35.311361 -80.711222             25               31   
3  2026-09-27_15-05-39  35.309306 -80.718056             25               31   
4  2026-09-27_15-05-39  35.306917 -80.722639             25               31   

  current_delay  confidence  
0          None           1  
1          None           1  
2          None           1  
3          None           1  
4          None           1  


In [ ]:
#Gets the data type for each attribute
df.dtypes

timestamp          datetime64[ns]
latitude                  float64
longitude                 float64
current_speed               int64
free_flow_speed             int64
current_delay              object
confidence                  int64
day_of_week                object
dtype: object

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
# Create time features
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

df.head(11)

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month
0,2026-09-27 15:05:39,35.307778,-80.724583,27,27,None,1,6,15,9
1,2026-09-27 15:05:39,35.313611,-80.701944,40,40,None,1,6,15,9
2,2026-09-27 15:05:39,35.311361,-80.711222,25,31,None,1,6,15,9
3,2026-09-27 15:05:39,35.309306,-80.718056,25,31,None,1,6,15,9
4,2026-09-27 15:05:39,35.306917,-80.722639,25,31,None,1,6,15,9
5,2026-09-27 15:05:40,35.293111,-80.749361,18,29,None,1,6,15,9
6,2026-09-27 15:05:40,35.294889,-80.742111,55,55,None,1,6,15,9
7,2026-09-27 15:05:40,35.299639,-80.745750,28,28,None,1,6,15,9
8,2026-09-27 15:05:40,35.287972,-80.759472,25,25,None,1,6,15,9
9,2026-09-27 15:05:40,35.301861,-80.752333,27,27,None,1,6,15,9


In [ ]:
# I believe this has been done below after you made the holidays remove and put in google doc please
#converting into a csv to save into our shared s3 bucket
df.to_csv("TomTom_traffic.csv", index=False)


s3_client.upload_file(
    "TomTom_traffic.csv",
    "s3-3601-final-project",
    "tomtom/traffic/speeds1.json.gz"
)

In [ ]:
# Remove and paste into google doc please
compressed = io.BytesIO()
with gzip.GzipFile(fileobj=compressed, mode="wb") as gz:
    gz.write(json.dumps(data_list).encode("utf-8"))
                                    
s3_client.put_object(
    Bucket=s3_bucket,
    Key=s3_key,
    Body=compressed.getvalue(),
    ContentType="application/json",
    ContentEncoding="gzip",
)

NameError: name 'compressed' is not defined

In [ ]:
# Calculate the speed ratio for each traffic record
df["speed_ratio"] = (df["current_speed"] / df["free_flow_speed"])

In [ ]:
calendar = USFederalHolidayCalendar()
    #To use after getting traffic data from holidays
holiday_names = calendar.holidays(
        start=df["timestamp"].min().normalize(),
        end=df["timestamp"].max().normalize(),
        return_name=True,
    )

df["holiday"] = (
        #map timestamp to the correct holiday name
        df["timestamp"].dt.normalize()
        .map(holiday_names)
        .fillna("Not a federal holiday")
    )
if not df.empty:
    holiday_names = calendar.holidays(
        start=df["timestamp"].min().normalize(),
        end=df["timestamp"].max().normalize(),
        return_name=True,
    )

    df["holiday"] = (
        df["timestamp"].dt.normalize()
        .map(holiday_names)
        .fillna("Not a federal holiday")
    )
else:
    print("No traffic rows available.")

In [ ]:
# KEEP ME 
from io import StringIO


s3_client = boto3.client("s3")

csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False)

s3_client.put_object(
    Bucket=s3_bucket, Key=s3_key, Body=csv_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': 'R08XAEEP1VB92Y1R',
  'HostId': 'Kfn9q5H2glrI+hbvvTzP8Ipy4NfaKCnIwFe4bhTd5TbXVORniICSf8Mct+N04WXdqKVmb9FUNis=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'Kfn9q5H2glrI+hbvvTzP8Ipy4NfaKCnIwFe4bhTd5TbXVORniICSf8Mct+N04WXdqKVmb9FUNis=',
   'x-amz-request-id': 'R08XAEEP1VB92Y1R',
   'date': 'Sun, 27 Sep 2026 15:06:02 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"da5ba3adb342c9750b10da4020cd961e"',
   'x-amz-checksum-crc32': 'hB/PPA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"da5ba3adb342c9750b10da4020cd961e"',
 'ChecksumCRC32': 'hB/PPA==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}